<a href="https://colab.research.google.com/github/Takashi-sketch/hello-world/blob/main/jquants-api-quick-start-v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# J-Quants API V2 Quick Startガイド

In [1]:
#@title 初期設定＆各種import
#@markdown ←にある▶ボタンを押すとGoogle Colabでコードを実行することができます。

#@markdown まずはこちらのコードを実行し、J-Quants APIを利用するため必要なパッケージをimportしましょう。

import json
import sys
import requests

from IPython.display import display
import pandas as pd

pd.set_option("display.max_columns", None)

API_URL = "https://api.jquants.com/v2"

## Step1：API利用開始までの流れ　※初回のみ実施

J-Quants APIのご利用を検討いただき、ありがとうございます。

**J-Quants APIをご利用いただくには、以下の2つを事前に行っていただく必要がございます。**
 1. [J-Quants Webサイト](https://jpx-jquants.com/)への登録
 2. J-Quants API利用のためのプラン（Free, Light, Standard, Premium）選択

まだ、ご登録もしくはプラン選択がお済みでない方は、まず上記の2項目を行っていただきますようお願いいたします。  
より具体的な手順は[こちら](https://jpx-jquants.com/ja/spec/quickstart)をご参照ください。

## Step2：APIキー取得

V2ではAPIキー方式による認証を採用しています。

APIキーはJ-Quants Webサイトのダッシュボードページから取得できます。

1. [J-Quants Webサイト](https://jpx-jquants.com/)にログイン
2. ダッシュボードページにアクセス
3. APIキーを取得（APIキーに有効期限はありません）

In [5]:
#@title **APIキーを設定**
#@markdown J-Quants WebサイトのダッシュボードにてAPIキーを取得し、以下に貼り付けてください。

api_key = "x5lSKIv88LaKCSpmsnnApdT6AE4c-P__0NZyH5JhdD4"#@param {type: "string"}

if api_key:
    headers = {"x-api-key": api_key}
    print("APIキーが設定されました。J-Quants APIを利用する準備が完了しました。")
else:
    print("APIキーを入力してください。")

APIキーが設定されました。J-Quants APIを利用する準備が完了しました。


## Step3：取得したAPIキーを用いて各APIをご利用ください。

### Freeプラン以上のプランで利用できるAPI
- 上場銘柄一覧（/equities/master）
- 株価四本値*（/equities/bars/daily）
- 財務情報（/fins/summary）
- 決算発表予定日（/fins/earnings-date）
- 決算発表予定日（3・9月期決算会社のみ）（/equities/earnings-calendar）
- 取引カレンダー（/markets/calendar）

\* プレミアムプランのユーザのみ、前後場の四本値及び取引高・取引代金の情報が取得可能


In [8]:
#@title 上場銘柄一覧（/equities/master）

#@markdown - 過去時点での銘柄情報、当日の銘柄情報および翌営業日時点の銘柄情報が取得可能です。
#@markdown - データの取得では、銘柄コード（code）または日付（date）の指定が可能です。

#@markdown （データ更新時刻）
#@markdown - 毎営業日の17:30頃、翌営業日の8:00頃

code = "7203"#@param {type:"string"}
date = ""#@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date

res = requests.get(f"{API_URL}/equities/master", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/equities/master", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

,Date,Code,CoName,CoNameEn,S17,S17Nm,S33,S33Nm,ScaleCat,Mkt,MktNm,Mrgn,MrgnNm,ProdCat
0,2026-05-29,72030,トヨタ自動車,TOYOTA MOTOR CORPORATION,6,自動車・輸送機,3700,輸送用機器,TOPIX Core30,0111,プライム,2,貸借,011


In [9]:
#@title 株価四本値（/equities/bars/daily）

#@markdown - 株価は分割・併合を考慮した調整済み株価（小数点第２位四捨五入）と調整前の株価を取得することができます。
#@markdown - データの取得では、銘柄コード（code）または日付（date）の指定が必須となります。

#@markdown （データ更新時刻）
#@markdown - 毎営業日の17:00頃

#@markdown - Premiumプランの方には、日通しに加え、前場(Morning)及び後場(Afternoon)の四本値及び取引高（調整前・後両方）・取引代金が取得可能です。


code = "7203"#@param {type:"string"}
date = ""#@param {type:"string"}
from_ = "" #@param {type:"string"}
to = "" #@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date
if from_ != "":
  params["from"] = from_
if to != "":
  params["to"] = to

res = requests.get(f"{API_URL}/equities/bars/daily", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/equities/bars/daily", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

,Date,Code,O,H,L,C,UL,LL,Vo,Va,AdjFactor,AdjO,AdjH,AdjL,AdjC,AdjVo,MktCap,ExRT
0,2024-05-31,72030,3370.0,3401.0,3348.0,3401.0,0,0,34664500.0,1.174995e+11,1.0,3370.0,3401.0,3348.0,3401.0,34664500.0,55487272.0,None
1,2024-06-03,72030,3402.0,3428.0,3321.0,3341.0,0,0,34205700.0,1.150886e+11,1.0,3402.0,3428.0,3321.0,3341.0,34205700.0,54508373.0,None
2,2024-06-04,72030,3300.0,3339.0,3284.0,3298.0,0,0,31397200.0,1.038681e+11,1.0,3300.0,3339.0,3284.0,3298.0,31397200.0,53806829.0,None
3,2024-06-05,72030,3250.0,3259.0,3203.0,3218.0,0,0,29958300.0,9.662745e+10,1.0,3250.0,3259.0,3203.0,3218.0,29958300.0,52501630.0,None
4,2024-06-06,72030,3258.0,3302.0,3249.0,3273.0,0,0,23424700.0,7.680427e+10,1.0,3258.0,3302.0,3249.0,3273.0,23424700.0,53398954.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
481,2026-05-25,72030,3051.0,3108.0,3006.0,3026.0,0,0,22249700.0,6.786559e+10,1.0,3051.0,3108.0,3006.0,3026.0,22249700.0,47795632.0,None
482,2026-05-26,72030,3028.0,3030.0,2989.0,3022.0,0,0,16349400.0,4.933089e+10,1.0,3028.0,3030.0,2989.0,3022.0,16349400.0,47732452.0,None
483,2026-05-27,72030,3016.0,3039.0,2998.5,3008.0,0,0,17980500.0,5.412794e+10,1.0,3016.0,3039.0,2998.5,3008.0,17980500.0,47511322.0,None
484,2026-05-28,72030,3039.0,3072.0,3023.0,3030.0,0,0,25312200.0,7.691564e+10,1.0,3039.0,3072.0,3023.0,3030.0,25312200.0,47858812.0,None


In [17]:
#@title 財務情報（/fins/summary）

#@markdown - 財務情報APIでは、上場企業がTDnetへ提出する決算短信Summary等を基に作成された、四半期毎の財務情報を取得することができます。
#@markdown - データの取得では、銘柄コード（code）または開示日（date）の指定が必須です。
#@markdown - `date` と組み合わせて `cursor` を使い回すことで、前回取得以降の新着財務情報のみを差分取得できます。Premiumプラン以上で利用可能。

#@markdown （データ更新時刻）
#@markdown - 速報18:00頃、確報24:30頃
#@markdown - Premiumプランは随時更新


code = "7203"#@param {type:"string"}
date = ""#@param {type:"string"}
cursor = ""#@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date
if cursor != "":
  params["cursor"] = cursor

all_data = []
cursor_value = None
pagination_key = None

while True:
  request_params = dict(params)
  if pagination_key:
    request_params["pagination_key"] = pagination_key

  res = requests.get(f"{API_URL}/fins/summary", params=request_params, headers=headers)

  if res.status_code != 200:
    print(res.json())
    break

  d = res.json()
  all_data.extend(d.get("data", []))

  if "cursor" in d:
    cursor_value = d["cursor"]

  pagination_key = d.get("pagination_key")
  if not pagination_key:
    break

if all_data:
  df = pd.DataFrame(all_data)
  display(df)

if cursor_value:
  print(f"\ncursor（次回取得用）: {cursor_value}")

,DiscDate,DiscTime,Code,DiscNo,DocType,CurPerType,CurPerSt,CurPerEn,CurFYSt,CurFYEn,NxtFYSt,NxtFYEn,Sales,OP,OdP,NP,EPS,DEPS,TA,Eq,EqAR,BPS,CFO,CFI,CFF,CashEq,Div1Q,Div2Q,Div3Q,DivFY,DivAnn,DivUnit,DivTotalAnn,PayoutRatioAnn,FDiv1Q,FDiv2Q,FDiv3Q,FDivFY,FDivAnn,FDivUnit,FDivTotalAnn,FPayoutRatioAnn,NxFDiv1Q,NxFDiv2Q,NxFDiv3Q,NxFDivFY,NxFDivAnn,NxFDivUnit,NxFPayoutRatioAnn,FSales2Q,FOP2Q,FOdP2Q,FNP2Q,FEPS2Q,NxFSales2Q,NxFOP2Q,NxFOdP2Q,NxFNp2Q,NxFEPS2Q,FSales,FOP,FOdP,FNP,FEPS,NxFSales,NxFOP,NxFOdP,NxFNp,NxFEPS,MatChgSub,SigChgInC,ChgByASRev,ChgNoASRev,ChgAcEst,RetroRst,ShOutFY,TrShFY,AvgSh,NCSales,NCOP,NCOdP,NCNP,NCEPS,NCTA,NCEq,NCEqAR,NCBPS,FNCSales2Q,FNCOP2Q,FNCOdP2Q,FNCNP2Q,FNCEPS2Q,NxFNCSales2Q,NxFNCOP2Q,NxFNCOdP2Q,NxFNCNP2Q,NxFNCEPS2Q,FNCSales,FNCOP,FNCOdP,FNCNP,FNCEPS,NxFNCSales,NxFNCOP,NxFNCOdP,NxFNCNP,NxFNCEPS,ShEq,NCShEq,ROE,NCROE
0,2024-08-01,13:25:00,72030,20240724553908,1QFinancialStatements_Consolidated_IFRS,1Q,2024-04-01,2024-06-30,2024-04-01,2025-03-31,,,11837879000000,1308462000000,,1333347000000,98.99,98.99,94037319000000,36779372000000,0.38,,683661000000,-2399603000000,-318790000000,7597094000000,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,46000000000000,4300000000000,,3570000000000,265.04,,,,,,,false,false,false,false,,15794987460,2325417265,13469159202,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,35737743000000,,,
1,2024-11-06,13:55:00,72030,20241101509817,2QFinancialStatements_Consolidated_IFRS,2Q,2024-04-01,2024-09-30,2024-04-01,2025-03-31,,,23282450000000,2464217000000,,1907113000000,142.15,142.15,89169296000000,35266663000000,0.385,,1817177000000,-3085752000000,-289752000000,7631457000000,,40.0,,,,,,,,,,50.0,90.0,,,,,,,,,,,,,,,,,,,,,46000000000000,4300000000000,,3570000000000,268.77,,,,,,,false,false,false,false,,15794987460,2645215176,13416064614,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,34368513000000,,,
2,2025-02-05,13:25:00,72030,20250131559581,3QFinancialStatements_Consolidated_IFRS,3Q,2024-04-01,2024-12-31,2024-04-01,2025-03-31,,,35673545000000,3679491000000,,4100389000000,307.95,307.95,94674416000000,36856527000000,0.379,,2823725000000,-3523253000000,-472090000000,8285156000000,,40.0,,,,,,,,,,50.0,90.0,,,,,,,,,,,,,,,,,,,,,47000000000000,4700000000000,,4520000000000,340.87,,,,,,,false,false,false,false,,15794987460,2699079277,13314911412,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,35910271000000,,,
3,2025-05-08,13:55:00,72030,20250425524402,FYFinancialStatements_Consolidated_IFRS,FY,2024-04-01,2025-03-31,2024-04-01,2025-03-31,2025-04-01,2026-03-31,48036704000000,4795586000000,,4765086000000,359.56,359.56,93601350000000,36878913000000,0.384,2753.09,3696934000000,-4189736000000,197236000000,8982404000000,,40.0,,50.0,90.0,,1178437000000,0.25,,,,,,,,,,45.0,,50.0,95.0,,0.399,,,,,,,,,,,,,,,,48500000000000,3800000000000,,3100000000000,237.57,,true,false,false,false,,15794987460,2746057686,13252455897,18277671000000,2966857000000,4538037000000,3845213000000,290.15,29046699000000,21504804000000,0.74,1648.01,,,,,,,,,,,,,,,,,,,,,35924826000000,21504804000000,0.136,
4,2025-08-07,14:00:00,72030,20250729522887,1QFinancialStatements_Consolidated_IFRS,1Q,2025-04-01,2025-06-30,2025-04-01,2026-03-31,,,12253326000000,1166141000000,,841345000000,64.56,64.56,93468143000000,36993052000000,0.386,,1876481000000,-1802002000000,-803284000000,8210856000000,,,,,,,,,,45.0,,50.0,95.0,,,,,,,,,,,,,,,,,,,,,48500000000000,3200000000000,,2660000000000,204.09,,,,,,,false,false,false,false,,15794987460,2761596216,13032932250,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,36040203000000,,,
5,2025-11-05,14:25:00,72030,20251029581373,2QFinancialStatements_Consolidated_IFRS,2Q,2025-04-01,2025-09-30,2025-04-01,2026-03-31,,,24630753000000,2005692000000,,1773426000000,136.07,136.07,97574878000000,38456954000000,0.384,,2944609000000,-3517528000000,-362065000000,8112922000000,,45.0,,,,,,,,,,50.0,95.0,,,,,,,,,,,,,,,,,,,,,49000000000000,3400000000000,,2930000000000,224.81,,,,,,,false,false,false,false,,15794987460,2761598241,13033161110,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,37492119000000,,,
6,2026-02-06,14:00:00,72030,2026020

In [13]:
#@title 決算発表予定日（/fins/earnings-date）

#@markdown - 東証上場会社等が東証に対して報告した決算発表予定日を取得できます。
#@markdown - 決算期によらず、報告を行った全上場銘柄（REIT等を含む）が対象で、予定日の変更・未定の履歴も含めて公表日単位で提供されます。
#@markdown - データの取得では、銘柄コード（code）・公表日（date）・発表予定日（scheduled_date）のいずれか1つの指定が必須です（2つ以上の同時指定は不可）。
#@markdown - 予定日が未定の場合、SchDate は空文字（""）で返却されます。

#@markdown （データ更新時刻）
#@markdown - 毎営業日の10:05頃

code = "7203"#@param {type:"string"}
date = ""#@param {type:"string"}
scheduled_date = ""#@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date
if scheduled_date != "":
  params["scheduled_date"] = scheduled_date

res = requests.get(f"{API_URL}/fins/earnings-date", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/fins/earnings-date", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

,PubDate,SchDate,FQName,FYE,Code,CoName,CoNameEn
0,2024-07-01,2024-08-01,1Q,0331,72030,トヨタ自動車,TOYOTA MOTOR CORPORATION
1,2024-09-30,2024-11-06,2Q,0331,72030,トヨタ自動車,TOYOTA MOTOR CORPORATION
2,2024-12-27,2025-02-05,3Q,0331,72030,トヨタ自動車,TOYOTA MOTOR CORPORATION
3,2025-03-31,2025-05-08,FY,0331,72030,トヨタ自動車,TOYOTA MOTOR CORPORATION
4,2025-06-30,2025-08-07,1Q,0331,72030,トヨタ自動車,TOYOTA MOTOR CORPORATION
5,2025-10-01,2025-11-05,2Q,0331,72030,トヨタ自動車,TOYOTA MOTOR CORPORATION
6,2025-12-26,2026-02-06,3Q,0331,72030,トヨタ自動車,TOYOTA MOTOR CORPORATION
7,2026-03-30,2026-05-08,FY,0331,72030,トヨタ自動車,TOYOTA MOTOR CORPORATION


In [14]:
#@title 決算発表予定日（3・9月期決算会社のみ）（/equities/earnings-calendar）

#@markdown （データ更新時刻）
#@markdown - 不定期（更新がある日は）19:00頃

#@markdown - [当該ページ](https://www.jpx.co.jp/listing/event-schedules/financial-announcement/index.html)で、3月期・９月期決算会社分に更新があった場合のみ19時ごろに更新されます。

params = {}

res = requests.get(f"{API_URL}/equities/earnings-calendar", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/equities/earnings-calendar", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

,Date,Code,CoName,FY,SectorNm,FQ,Section
0,2026-08-17,46510,サニックスホールディングス,3月31日,サービス業,第１四半期,スタンダード


In [15]:
#@title 取引カレンダー（/markets/calendar）

#@markdown - 東証およびOSEにおける営業日、休業日、ならびにOSEにおける祝日取引の有無の情報を取得できます。
#@markdown - データの取得では、休日区分（hol_div）または日付（from/to）の指定が可能です。

#@markdown （データ更新日）
#@markdown - 不定期（原則として、毎年2月頃をめどに翌年1年間の営業日および祝日取引実施日（予定）を更新します。）


hol_div = ""#@param ["0", "1", "2", "3"]{allow-input:true}
from_ = "" #@param {type:"string"}
to = "" #@param {type:"string"}

params = {}
if hol_div != "":
  params["hol_div"] = hol_div
if from_ != "":
  params["from"] = from_
if to != "":
  params["to"] = to

res = requests.get(f"{API_URL}/markets/calendar", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/markets/calendar", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

,Date,HolDiv
0,2024-05-31,1
1,2024-06-01,0
2,2024-06-02,0
3,2024-06-03,1
4,2024-06-04,1
...,...,...
726,2026-05-27,1
727,2026-05-28,1
728,2026-05-29,1
729,2026-05-30,0


### Lightプラン以上のプランで利用できるAPI
- 投資部門別情報（/equities/investor-types）
- TOPIX四本値（/indices/bars/daily/topix）

In [ ]:
#@title 投資部門別（株式）データ（/equities/investor-types）

#@markdown - 投資部門別売買状況（金額）のデータを取得することができます。
#@markdown - 投資部門別売買状況は、個人・外国人・金融機関など、投資家ごとの売買動向をまとめた情報です。
#@markdown - 基本的には[こちら](https://www.jpx.co.jp/markets/statistics-equities/investor-type/index.html)のページで掲載しているものと同等のものになります。
#@markdown - データの取得では、セクション（section）または日付（from/to）の指定が可能です。

#@markdown （データ更新時刻）
#@markdown - 原則、毎週第４営業日18:00頃


section = ""#@param ["TSE1st", "TSE2nd", "TSEMothers", "TSEJASDAQ", "TSEPrime", "TSEStandard", "TSEGrowth", "TokyoNagoya"] {allow-input: true}
from_ = "26/08/04" #@param {type:"string"}
to = "" #@param {type:"string"}

params = {}
if section != "":
  params["section"] = section
if from_ != "":
  params["from"] = from_
if to != "":
  params["to"] = to

res = requests.get(f"{API_URL}/equities/investor-types", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/equities/investor-types", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())


In [ ]:
#@title TOPIX四本値（/indices/bars/daily/topix）

#@markdown - 日次のTOPIX指数の四本値について取得することができます。
#@markdown - 日付（from/to）の指定が可能です

#@markdown （データ更新時刻）
#@markdown - 毎営業日の16:30頃


date_from = ""#@param {type:"string"}
date_to = ""#@param {type:"string"}

params = {}

if date_from != "":
  params["from"] = date_from
if date_to != "":
  params["to"] = date_to

res = requests.get(f"{API_URL}/indices/bars/daily/topix", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/indices/bars/daily/topix", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())


### Standard以上のプランで利用できるAPI
- 指数四本値（/indices/bars/daily）
- 日経225オプション四本値（/derivatives/bars/daily/options/225）
- 信用取引週末残高（/markets/margin-interest）
- 業種別空売り比率（/markets/short-ratio）
- 空売り残高報告（/markets/short-sale-report）
- 日々公表信用取引残高（/markets/margin-alert）
- 大株主状況（EDINET）（/edinet/major-shareholders）
- 政策保有株式（EDINET）（/edinet/cross-shareholdings）
- 大量保有報告書（EDINET）（/edinet/large-volume-shareholders）

In [ ]:
#@title 指数四本値（/indices/bars/daily）

#@markdown - 日次の各種指数の四本値について取得することができます。
#@markdown - データの取得では、指数コード（code）または日付（date）の指定が必須となります。

#@markdown （データ更新時刻）
#@markdown - 毎営業日の16:30頃


code = ""#@param {type:"string"}
date = ""#@param {type:"string"}
from_ = "" #@param {type:"string"}
to = "" #@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date
if from_ != "":
  params["from"] = from_
if to != "":
  params["to"] = to

res = requests.get(f"{API_URL}/indices/bars/daily", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/indices/bars/daily", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

In [ ]:
#@title 日経225オプション四本値（/derivatives/bars/daily/options/225）

#@markdown - 日次の日経225指数オプションの四本値や売買高、清算値段等について取得することができます。
#@markdown - 日付（date）の指定が必須です

#@markdown （データ更新時刻）
#@markdown - 毎営業日の27:00頃


date = "" #@param {type:"string"}
params = {}
if date != "":
  params["date"] = date

res = requests.get(f"{API_URL}/derivatives/bars/daily/options/225", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/derivatives/bars/daily/options/225", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

In [ ]:
#@title 信用取引週末残高（/markets/margin-interest）

#@markdown - 制度・一般信用取引における各銘柄の前週末の残高についてデータを取得することができます。
#@markdown - 本データは[こちら](https://www.jpx.co.jp/markets/statistics-equities/margin/index.html)の個別銘柄信用取引残高表のデータをヒストリカルで提供するものになります。
#@markdown - 銘柄コード（code）もしくは日付（date）の指定が必須です。
#@markdown （From, Toはcodeを入れている場合にのみ可能）

#@markdown （データ更新時刻）
#@markdown - 原則毎週第２営業日の16:30頃


code = ""#@param {type:"string"}
date = ""#@param {type:"string"}
from_ = "" #@param {type:"string"}
to = "" #@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date
if from_ != "":
  params["from"] = from_
if to != "":
  params["to"] = to

res = requests.get(f"{API_URL}/markets/margin-interest", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/markets/margin-interest", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

In [ ]:
#@title 業種別空売り比率（/markets/short-ratio）

#@markdown - 日次の３３業種別の空売りの売買代金について取得することができます。
#@markdown - ３３業種コード（s33）もしくは日付（date）の指定が必須です。
#@markdown （From, Toはs33を入れている場合にのみ可能）

#@markdown （データ更新時刻）
#@markdown - 毎営業日の16:30頃

#@markdown （その他留意点）
#@markdown - 業種コード9999は、ETFやREIT等の３３業種に含まれない銘柄のものになります。


s33 = ""#@param {type:"string"}
date = ""#@param {type:"string"}
from_ = "" #@param {type:"string"}
to = "" #@param {type:"string"}

params = {}
if s33 != "":
  params["s33"] = s33
if date != "":
  params["date"] = date
if from_ != "":
  params["from"] = from_
if to != "":
  params["to"] = to

res = requests.get(f"{API_URL}/markets/short-ratio", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/markets/short-ratio", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

In [ ]:
#@title 空売り残高報告（/markets/short-sale-report）

#@markdown - 「有価証券の取引等の規制に関する内閣府令」に基づき、取引参加者より報告を受けたもののうち、残高割合が0.5％以上のものについての情報を取得できます。
#@markdown - データの取得では、銘柄コード（code）、公表日（disc_date）、計算日（calc_date）のいずれかの指定が必須となります。

#@markdown （データ更新時刻）
#@markdown - 毎営業日の17:30頃

code = ""#@param {type:"string"}
disc_date = ""#@param {type:"string"}
disc_date_from = ""#@param {type:"string"}
disc_date_to = ""#@param {type:"string"}
calc_date = ""#@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if disc_date != "":
  params["disc_date"] = disc_date
if disc_date_from != "":
  params["disc_date_from"] = disc_date_from
if disc_date_to != "":
  params["disc_date_to"] = disc_date_to
if calc_date != "":
  params["calc_date"] = calc_date

res = requests.get(f"{API_URL}/markets/short-sale-report", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/markets/short-sale-report", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

In [ ]:
#@title 日々公表信用取引残高（/markets/margin-alert）

#@markdown - 東京証券取引所または日本証券金融が、日次の信用取引残高を公表する必要があると認めた銘柄のみが収録されます。
#@markdown - データの取得では、銘柄コード（code）、公表日（date）のいずれかの指定が必須となります。

#@markdown （データ更新時刻）
#@markdown - 毎営業日の16:30頃

code = ""#@param {type:"string"}
_date = ""#@param {type:"string"}
_from = ""#@param {type:"string"}
_to = ""#@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if _date != "":
  params["date"] = _date
if _from != "":
  params["from"] = _from
if _to != "":
  params["to"] = _to

res = requests.get(f"{API_URL}/markets/margin-alert", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/markets/margin-alert", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

In [ ]:
#@title 大株主状況（EDINET）（/edinet/major-shareholders）

#@markdown - 有価証券報告書に記載されている大株主の状況を取得できます。
#@markdown - edinet_code / code / date は任意指定です（edinet_code と code の同時指定は不可）。すべて省略した場合は、API実行日に提出された全有報のデータが対象となります。
#@markdown - 大株主レコードは Hldrs 列にネストされた配列として返却されます（pd.json_normalize で明細を展開する例も下記に含みます）。

#@markdown （データ更新時刻）
#@markdown - 随時更新（平日 8:00〜17:59、開示情報が発生次第、順次反映）

edinet_code = ""#@param {type:"string"}
code = "7203"#@param {type:"string"}
date = ""#@param {type:"string"}

params = {}
if edinet_code != "":
  params["edinet_code"] = edinet_code
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date

res = requests.get(f"{API_URL}/edinet/major-shareholders", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/edinet/major-shareholders", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

if res.status_code == 200 and data:
  # 大株主明細（Hldrs）を展開して表示する例
  df_holders = pd.json_normalize(data, record_path="Hldrs", meta=["DocId", "Code", "SubDate"])
  display(df_holders)

In [ ]:
#@title 政策保有株式（EDINET）（/edinet/cross-shareholdings）

#@markdown - 有価証券報告書「株式の保有状況」に記載されている政策保有株式を取得できます。
#@markdown - edinet_code / code / date は任意指定です（edinet_code と code の同時指定は不可）。すべて省略した場合は、API実行日に提出された全有報のデータが対象となります。
#@markdown - 保有主体ブロックは Report / Largest / SecondLargest 列にネストされたオブジェクトとして返却されます（内部に特定投資株式 Spec / みなし保有株式 Deem の銘柄明細を含みます）。

#@markdown （データ更新時刻）
#@markdown - 随時更新（平日 8:00〜17:59、開示情報が発生次第、順次反映）

edinet_code = ""#@param {type:"string"}
code = "7203"#@param {type:"string"}
date = ""#@param {type:"string"}

params = {}
if edinet_code != "":
  params["edinet_code"] = edinet_code
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date

res = requests.get(f"{API_URL}/edinet/cross-shareholdings", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/edinet/cross-shareholdings", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

In [ ]:
#@title 大量保有報告書（EDINET）（/edinet/large-volume-shareholders）

#@markdown - 大量保有報告書・変更報告書に記載されている発行者、提出者情報を取得できます。
#@markdown - edinet_code / code / date は任意指定です（edinet_code と code の同時指定は不可）。すべて省略した場合は、API実行日に提出された全書類のデータが対象となります。
#@markdown - 提出者及び共同保有者のレコードは Hldrs 列にネストされた配列として返却されます。

#@markdown （データ更新時刻）
#@markdown - 随時更新（平日 8:00〜17:59、開示情報が発生次第、順次反映）

edinet_code = ""#@param {type:"string"}
code = "7203"#@param {type:"string"}
date = ""#@param {type:"string"}

params = {}
if edinet_code != "":
  params["edinet_code"] = edinet_code
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date

res = requests.get(f"{API_URL}/edinet/large-volume-shareholders", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/edinet/large-volume-shareholders", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

### Premiumプランで利用できるAPI
- 売買内訳データ（/markets/breakdown）
- 前場四本値（/equities/bars/daily/am）
- 配当金情報（/fins/dividend）
- 財務諸表(BS/PL/CF)（/fins/details）
- 先物四本値（/derivatives/bars/daily/futures）
- オプション四本値（/derivatives/bars/daily/options）

In [ ]:
#@title 売買内訳データ（/markets/breakdown）

#@markdown - 売買内訳データは、東証上場銘柄の東証市場における銘柄別の日次売買代金・売買高(立会内取引に限る)について、信用取引や空売りの利用に関して、発注時のフラグ情報を用いて細分化したデータです。
#@markdown - 銘柄コード（code）もしくは日付（date）の指定が必須です。
#@markdown （From, Toはcodeを入れている場合にのみ可能）

#@markdown （データ更新時刻）
#@markdown - 毎営業日の18:00頃

#@markdown （その他留意点）
#@markdown - 当日に立会内取引が成立しなかった場合（約定なし）、当該銘柄はレコードに含まれていません。

code = ""#@param {type:"string"}
date = ""#@param {type:"string"}
from_ = "" #@param {type:"string"}
to = "" #@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date
if from_ != "":
  params["from"] = from_
if to != "":
  params["to"] = to

res = requests.get(f"{API_URL}/markets/breakdown", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/markets/breakdown", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

In [ ]:
#@title 前場四本値（/equities/bars/daily/am）

#@markdown - 各銘柄の前場の四本値及び取引高・代金について、当日の前場終了後のデータを取得できます。
#@markdown - 銘柄コード（code）を指定することができます。codeパラメータがない場合は、全銘柄取得されます。

#@markdown （データ更新時刻）
#@markdown - 毎営業日の12:00頃

#@markdown （その他留意点）
#@markdown - 本APIのデータは当日12:00頃〜翌日の朝6:00頃まで取得可能です。
#@markdown - 当日以外のヒストリカルのデータは、Premiumユーザの方は株価四本値（/equities/bars/daily）のAPIで取得可能です。
#@markdown - なお、上記以外の時間にAPIコールした場合は、StatusCode = 210が返却されます。（通常は200）


code = ""#@param {type:"string"}

params = {}
if code != "":
  params["code"] = code

res = requests.get(f"{API_URL}/equities/bars/daily/am", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/equities/bars/daily/am", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

In [ ]:
#@title 配当金情報（/fins/dividend）

#@markdown - 上場会社の配当（決定・予想）に関する１株当たり配当金額、基準日、権利落日及び支払開始予定日等の情報が取得できます。
#@markdown - 銘柄コード（code）もしくは日付（date）の指定が必須です。
#@markdown （From, Toはcodeを入れている場合にのみ可能）

#@markdown （データ更新時刻）
#@markdown - 毎営業日の19:00頃


code = "7203"#@param {type:"string"}
date = "26/08/04"#@param {type:"string"}
from_ = "" #@param {type:"string"}
to = "" #@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date
if from_ != "":
  params["from"] = from_
if to != "":
  params["to"] = to

res = requests.get(f"{API_URL}/fins/dividend", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/fins/dividend", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

In [ ]:
#@title 財務諸表(BS/PL/CF)（/fins/details）

#@markdown - 財務諸表(BS/PL/CF)APIでは、上場企業の四半期毎の財務情報における、貸借対照表、損益計算書に記載の項目を取得することができます。
#@markdown - 本コードでは、出力結果の表示方法をformat_typeを指定して切り替え可能です。flat:全項目をヘッダーに表示する、non-flat:詳細項目を1列に集約します。
#@markdown - データの取得では、銘柄コード（code）または開示日（date）の指定が必須です。
#@markdown - `date` と組み合わせて `cursor` を使い回すことで、前回取得以降の新着財務情報のみを差分取得できます。

#@markdown （データ更新時刻）
#@markdown - 速報18:00頃、確報24:30頃


format_type = "non-flat"#@param ["non-flat", "flat"]
code = "7203"#@param {type:"string"}
date = "26/08/04"#@param {type:"string"}
cursor = ""#@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date
if cursor != "":
  params["cursor"] = cursor

all_data = []
cursor_value = None
pagination_key = None

while True:
  request_params = dict(params)
  if pagination_key:
    request_params["pagination_key"] = pagination_key

  res = requests.get(f"{API_URL}/fins/details", params=request_params, headers=headers)

  if res.status_code != 200:
    print(res.json())
    break

  d = res.json()
  all_data.extend(d.get("data", []))

  if "cursor" in d:
    cursor_value = d["cursor"]

  pagination_key = d.get("pagination_key")
  if not pagination_key:
    break

if all_data:
  if format_type == "non-flat":
    df = pd.DataFrame(all_data)
  else:
    df = pd.json_normalize(all_data)
  display(df)

if cursor_value:
  print(f"\ncursor（次回取得用）: {cursor_value}")

In [ ]:
#@title 先物四本値（/derivatives/bars/daily/futures）

#@markdown - 先物四本値APIでは、先物に関する、四本値や清算値段、理論価格に関する情報を取得することができます。
#@markdown - データの取得では、日付（date）の指定が必須です。

#@markdown （データ更新時刻）
#@markdown - 毎営業日の27:00頃

category = ""#@param {type:"string"}
date = ""#@param {type:"string"}
contract_flag = ""#@param{type:"string"}

params = {}
if category != "":
  params["category"] = category
if date != "":
  params["date"] = date
if contract_flag != "":
  params["contract_flag"] = contract_flag

res = requests.get(f"{API_URL}/derivatives/bars/daily/futures", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/derivatives/bars/daily/futures", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

In [ ]:
#@title オプション四本値（/derivatives/bars/daily/options）

#@markdown - オプション四本値APIでは、オプションに関する、四本値や清算値段、理論価格に関する情報を取得することができます。
#@markdown - データの取得では、日付（date）の指定が必須です。

#@markdown （データ更新時刻）
#@markdown - 毎営業日の27:00頃

category = ""#@param {type:"string"}
date = ""#@param {type:"string"}
contract_flag = ""#@param{type:"string"}

params = {}
if category != "":
  params["category"] = category
if date != "":
  params["date"] = date
if contract_flag != "":
  params["contract_flag"] = contract_flag

res = requests.get(f"{API_URL}/derivatives/bars/daily/options", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/derivatives/bars/daily/options", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

### Bulk API（CSVダウンロード）
- ダウンロード可能ファイル一覧（/bulk/list）
- ファイルダウンロード用URL取得（/bulk/get）

Bulk APIを使用すると、CSV形式で大量のデータを効率的にダウンロードできます。
**Lightプラン以上**で利用できます（Freeプランでは取引カレンダー（/markets/calendar）のCSVのみ取得可能です）。


In [ ]:
#@title ダウンロード可能ファイル一覧（/bulk/list）

#@markdown - CSV形式でダウンロード可能なファイルの一覧を取得するエンドポイントです。
#@markdown - endpointパラメータで取得したいデータセットを指定します。

#@markdown （指定可能なendpoint例）
#@markdown - /equities/bars/daily（株価四本値）
#@markdown - /equities/bars/minute（株価分足）
#@markdown - /fins/summary（財務情報）
#@markdown - /indices/bars/daily/topix（TOPIX指数四本値）
#@markdown - など

#@markdown 指定可能なendpointの完全な一覧は[こちら](https://jpx-jquants.com/ja/spec/bulk-list/endpoints)をご覧ください。

endpoint = "\t/markets/calendar"#@param {type:"string"}

params = {}
if endpoint != "":
  params["endpoint"] = endpoint

res = requests.get(f"{API_URL}/bulk/list", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())


In [ ]:
#@title ファイルダウンロード用URL取得（/bulk/get）

#@markdown - /bulk/list で取得したKeyを指定して、CSVファイルダウンロード用の署名付きURLを取得します。
#@markdown - 取得したURLの有効期限は約5分です。期限内にダウンロードを完了してください。

#@markdown （使い方）
#@markdown 1. まず /bulk/list でダウンロード可能なファイル一覧を取得
#@markdown 2. 取得したファイル一覧からダウンロードしたいファイルの Key を確認
#@markdown 3. その Key を下記に入力して実行
#@markdown 4. 取得したURLにアクセスしてCSVをダウンロード

key = ""#@param {type:"string"}

params = {}
if key != "":
  params["key"] = key

res = requests.get(f"{API_URL}/bulk/get", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  download_url = d.get("url", "")
  print("ダウンロードURL（有効期限: 約5分）:")
  print(download_url)
  print("\n※ 上記URLにアクセスしてCSVファイルをダウンロードしてください。")
else:
  print(res.json())


In [ ]:
#@title Bulk APIを組み合わせた実行例（list → get）

#@markdown - /bulk/list でファイル一覧を取得し、特定のファイルの Key を使って /bulk/get でダウンロードURLを取得する実行例です。

#@markdown （使い方）
#@markdown 1. endpoint に取得したいデータセットのエンドポイントを指定
#@markdown 2. num_files に取得したいファイルの件数を指定（デフォルト: 3件）
#@markdown 3. 実行するとファイル一覧が表示されます
#@markdown 4. 一覧の中から指定された件数分のファイルの Key を取得し、ダウンロードURLを取得

endpoint = "/equities/bars/daily"#@param {type:"string"}
num_files = 3#@param {type:"integer"}

# Step 1: ファイル一覧を取得
params = {"endpoint": endpoint}
res = requests.get(f"{API_URL}/bulk/list", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  files = d["data"]
  df_files = pd.DataFrame(files)

  print("=== 利用可能なファイル一覧 ===")
  display(df_files)

  if len(files) > 0:
    # 指定された件数分のファイルのKeyを取得（最大でファイル数まで）
    target_count = min(num_files, len(files))
    print(f"\n{target_count}件のファイルのダウンロードURLを取得します...")

    # Step 2: 各ファイルのダウンロードURLを取得
    for i in range(target_count):
      file_key = files[i]["Key"]
      print(f"\n--- ファイル {i+1}/{target_count} ---")
      print(f"Key: {file_key}")

      params_get = {"key": file_key}
      res_get = requests.get(f"{API_URL}/bulk/get", params=params_get, headers=headers)

      if res_get.status_code == 200:
        d_get = res_get.json()
        download_url = d_get.get("url", "")
        print(f"URL（有効期限: 約5分）:")
        print(download_url)
      else:
        print("ダウンロードURL取得エラー:")
        print(res_get.json())

    print("\n※ 上記URLにアクセスしてCSVファイルをダウンロードしてください。")
  else:
    print("\n利用可能なファイルがありません。")
else:
  print("ファイル一覧取得エラー:")
  print(res.json())

### Lightプラン以上 + アドオン契約で利用できるAPI（株価 分足・ティック）
- 株価分足（/equities/bars/minute）

\* 株価 分足・ティック アドオンの契約が必要です。データ取得可能期間は過去2年間です。


In [ ]:
#@title 株価分足（/equities/bars/minute）

#@markdown - 分足の株価データを取得するエンドポイント。1分単位の四本値（始値・高値・安値・終値）、出来高、売買代金のデータを取得できます。
#@markdown - データの取得では、銘柄コード（code）または日付（date）の指定が必須となります。
#@markdown - データ取得可能期間は過去2年間です。

#@markdown （データ更新時刻）
#@markdown - 毎営業日の16:30頃

#@markdown （その他留意点）
#@markdown - 株価 分足・ティック アドオンの契約が必要です。
#@markdown - 全銘柄のデータを一括取得する場合は、Bulk APIの使用を推奨します。

code = ""#@param {type:"string"}
date = "20250122"#@param {type:"string"}
from_ = "" #@param {type:"string"}
to = "" #@param {type:"string"}

params = {}
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date
if from_ != "":
  params["from"] = from_
if to != "":
  params["to"] = to

res = requests.get(f"{API_URL}/equities/bars/minute", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/equities/bars/minute", params=params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())


### Lightプラン以上 + アドオン契約で利用できるAPI（TDnet/適時開示情報）
- 適時開示インデックス一覧（/td/list）
- 適時開示ファイルダウンロードURL取得（/td/files）
- 適時開示インデックス一括ダウンロード（/td/bulk）

\* **TDnet/適時開示情報アドオン**の契約が必要です。データ取得可能期間は過去5年間です。


In [ ]:
#@title 適時開示インデックス一覧（/td/list）

#@markdown - 適時開示のインデックス情報（開示番号・日時・タイトルなど）の一覧を取得できます。
#@markdown - `date`（開示日）または `code`（銘柄コード）のいずれかを必ず指定してください。
#@markdown - `date` に当日を指定して `cursor` を使い回すことで、新着開示情報をセミリアルタイムに取得できます。

#@markdown （パラメータ例）
#@markdown - date: 取得したい開示日（例: 20250401）
#@markdown - code: 銘柄コード（例: 86970 または 8697）
#@markdown - from/to: code と組み合わせた期間指定（例: from=20250101 to=20250401）
#@markdown - discItems: 公開項目コードで絞り込み（カンマ区切り）
#@markdown - cursor: 前回レスポンスの cursor 値（date と組み合わせて使用、前回取得以降の新規開示を取得）

date = "20250401"#@param {type:"string"}
code = ""#@param {type:"string"}
from_date = ""#@param {type:"string"}
to_date = ""#@param {type:"string"}
disc_items = ""#@param {type:"string"}
cursor = ""#@param {type:"string"}

params = {}
if date != "":
  params["date"] = date
if code != "":
  params["code"] = code
if from_date != "":
  params["from"] = from_date
if to_date != "":
  params["to"] = to_date
if disc_items != "":
  params["discItems"] = disc_items
if cursor != "":
  params["cursor"] = cursor

all_data = []
cursor_value = None
pagination_key = None

while True:
  request_params = dict(params)
  if pagination_key:
    request_params["pagination_key"] = pagination_key

  res = requests.get(f"{API_URL}/td/list", params=request_params, headers=headers)

  if res.status_code != 200:
    print(res.json())
    break

  d = res.json()
  all_data.extend(d.get("data", []))

  if "cursor" in d:
    cursor_value = d["cursor"]

  pagination_key = d.get("pagination_key")
  if not pagination_key:
    break

if all_data:
  df = pd.DataFrame(all_data)
  display(df)

if cursor_value:
  print(f"\ncursor（次回取得用）: {cursor_value}")


In [ ]:
#@title 適時開示ファイルダウンロードURL取得（/td/files）

#@markdown - 開示番号（discNo）に対応するファイル（PDF / XBRL）のダウンロードURLを取得します。
#@markdown - /td/list で取得した DiscNo を discNo に入力してください。
#@markdown - 取得したURLの有効期限は15分です。

#@markdown （docs パラメータで取得ファイル種別を絞り込めます）
#@markdown - g: 全文情報PDF
#@markdown - s: サマリ情報PDF
#@markdown - x: XBRL関連ファイル
#@markdown - 省略時は全種類を返します（例: g,s,x）

disc_no = "20250401130100"#@param {type:"string"}
docs = ""#@param {type:"string"}

params = {}
if disc_no != "":
  params["discNo"] = disc_no
if docs != "":
  params["docs"] = docs

res = requests.get(f"{API_URL}/td/files", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  print(f"開示番号: {d.get('discNo')}")
  files = d.get("files")
  if files:
    for key, url in files.items():
      print(f"\n{key}（有効期限: 15分）:")
      print(url)
  else:
    print("ファイルが見つかりませんでした。")
else:
  print(res.json())


In [ ]:
#@title 適時開示インデックス一括ダウンロード（/td/bulk）

#@markdown - 過去5年分の適時開示インデックス情報をまとめたCSV（gzip圧縮）のダウンロードURLを取得します。
#@markdown - 取得したURLの有効期限は15分です。期限内にダウンロードを完了してください。

#@markdown （使い方）
#@markdown 1. 実行するとCSVファイルのダウンロードURLと最終更新日時が表示されます
#@markdown 2. 取得したURLにアクセスしてCSV（gzip形式）をダウンロード
#@markdown 3. CSVを解凍して開示情報を取得

import io
import gzip

res = requests.get(f"{API_URL}/td/bulk", headers=headers)

if res.status_code == 200:
  d = res.json()
  last_updated = d.get("lastUpdated", "")
  download_url = d.get("url", "")
  print(f"最終更新日時: {last_updated}")
  print(f"\nダウンロードURL（有効期限: 15分）:")
  print(download_url)

  # ダウンロードしてDataFrameに読み込む例
  if download_url:
    print("\nCSVをダウンロードしてDataFrameに読み込み中...")
    csv_res = requests.get(download_url)
    if csv_res.status_code == 200:
      with gzip.open(io.BytesIO(csv_res.content), "rt", encoding="utf-8") as f:
        df = pd.read_csv(f)
      print(f"取得件数: {len(df)} 件")
      display(df.head())
    else:
      print(f"ダウンロードエラー: {csv_res.status_code}")
else:
  print(res.json())
